# Frozen-text fusion: does text help without modality dropout?

Same model and code as the 0.874 frozen-text fusion run (15 log-sum-exp layer tokens + frozen BAAI/bge-base-en-v1.5 question/answer token states, 2-layer fusion transformer, 10 epochs, batch 128, BCE, same split and held-out TriviaQA evaluation). The only change is the training-time text dropout.

Configurations, each with seeds 42 / 43 / 44:
* **fused, 0% text dropout**: text is always present during training
* **fused, 30% text dropout**: the earlier setting
* **image only**: no text at all

If fused-0% beats image-only across seeds, the text carries information that the 30% dropout was suppressing; if not, the text is redundant with the hidden states.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout de3a98b
!pip install -q -r requirements-kaggle.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
CONFIGS = {'fused_drop0': ['--mode', 'fused', '--text-dropout', '0'], 'fused_drop30': ['--mode', 'fused', '--text-dropout', '0.3'], 'image': ['--mode', 'image']}
OUTPUT_DIRS = {}
for seed in (42, 43, 44):
    for name, extra in CONFIGS.items():
        output_dir = Path(f'/kaggle/working/frozen_text_{name}_s{seed}')
        OUTPUT_DIRS[(name, seed)] = output_dir
        command = [sys.executable, '/kaggle/working/SALA/sala_vlm_frozen_text.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(output_dir), '--bleurt-threshold', '0.5', '--epochs-erm', '10', '--seed', str(seed), '--train-device', 'cuda:0', '--batch-size', '128', '--image', 'logsumexp', *extra]
        print('Running:', ' '.join(command), flush=True)
        subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
rows = []
for (name, seed), output_dir in OUTPUT_DIRS.items():
    s = json.loads((output_dir / 'summary.json').read_text())
    rows.append({'config': name, 'seed': seed, 'best_epoch': s['best_epoch'],
                 'source_val_auroc': s['best_source_validation_auroc_during_training'],
                 'source_test_auroc': s['source_test_metrics']['auroc'],
                 'target_auroc': s['target_metrics']['auroc'],
                 'target_balanced_accuracy': s['target_metrics']['balanced_accuracy'],
                 'text_masked_target_auroc': s.get('text_masked_target_auroc'),
                 'image_masked_target_auroc': s.get('image_masked_target_auroc')})
table = pd.DataFrame(rows)
display(table)
summary = table.groupby('config')[['source_val_auroc', 'source_test_auroc', 'target_auroc', 'target_balanced_accuracy', 'text_masked_target_auroc', 'image_masked_target_auroc']].agg(['mean', 'std'])
display(summary)
print(summary.to_string())